# 3. FREAK (Fast Retina Keypoint) Pipeline
### Bio-Inspired Binary Descriptor for Object Identification & Detection

### What is FREAK?
**FREAK** (*Fast Retina Keypoint*) was introduced by Alahi et al. (CVPR 2012). It is a fast, robust **binary** descriptor inspired by the human visual system:
- **Retinal Sampling Grid**: Mimics the distribution of photoreceptors in the human retina (fovea), with a high density of receptive fields near the center and decreasing density toward the periphery.
- **Binary Descriptor**: 64 bytes (512 bits) of intensity difference tests, comparing pairs of retinal receptive fields.
- **Detector-Free Extractor**: FREAK **only computes descriptors**; it requires a separate detector (such as **FAST** or **BRISK**) to identify keypoints.
- **Hamming Distance**: Matching uses bitwise XOR (`cv2.NORM_HAMMING`), executing orders of magnitude faster than floating-point L2 norms!



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from dataset import load_split
from freak_pipeline import FREAKBoVWPipeline, FREAKFeatureMatcher

def show_rgb(img, title=None):
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img_rgb = img
    plt.figure(figsize=(7, 7))
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(title, fontsize=12)
    plt.show()



## Step 1: Detect Keypoints with FAST and Compute FREAK Descriptors


In [ ]:
train_imgs, train_lbls, classes, _ = load_split("train")
test_imgs, test_lbls, _, _ = load_split("test")

# FAST detector + FREAK descriptor
fast = cv2.FastFeatureDetector_create(threshold=25, nonmaxSuppression=True)
freak = cv2.xfeatures2d.FREAK_create()

sample_img = train_imgs[10]
gray = cv2.cvtColor(sample_img, cv2.COLOR_BGR2GRAY)
kps = fast.detect(gray, None)
kps, descs = freak.compute(gray, kps)

print(f"FAST Detected: {len(kps)} keypoints")
print(f"FREAK Descriptors Shape: {descs.shape} (dtype: {descs.dtype})")
print(f"Descriptor bit length: {descs.shape[1] * 8} bits (64 bytes)")

img_kps = cv2.drawKeypoints(sample_img, kps, None, color=(0, 255, 255), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show_rgb(img_kps, title="FAST Keypoints + FREAK Bio-Inspired Retinal Descriptors")



## Step 2: Method A - FREAK Bag of Visual Words (BoVW) for Object Identification


In [ ]:
freak_bovw = FREAKBoVWPipeline(detector_type="fast", max_keypoints=300, vocab_size=150)
print("Training FREAK BoVW Pipeline...")
info = freak_bovw.train(train_imgs, train_lbls, C=1.0)
print(f"Trained in {info['train_time_sec']:.2f} seconds!")



## Step 3: Evaluate FREAK BoVW on Full Test Set


In [ ]:
eval_res = freak_bovw.evaluate(test_imgs, test_lbls)
print(f"FREAK BoVW Test Accuracy: {eval_res['accuracy'] * 100:.2f}%")
print(f"Average Inference Latency: {eval_res['avg_time_per_image_ms']:.2f} ms/image
")
print(classification_report(test_lbls, eval_res['predictions'], target_names=freak_bovw.classes_))

# Plot confusion matrix
cm = np.array(eval_res["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Oranges")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("FREAK BoVW Confusion Matrix")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="blue" if i != j else "black")
plt.colorbar(im)
plt.show()



## Step 4: Method B - Binary Feature Matching (cv2.NORM_HAMMING) & Homography Detection


In [ ]:
freak_matcher = FREAKFeatureMatcher(detector_type="fast", max_keypoints=500, ratio_thresh=0.80)

f0 = cv2.imread("../data/train/people/02022_0954.MOV_Bielsko-Biała_Pride_frame_000.jpg")
f1 = cv2.imread("../data/train/people/02022_0954.MOV_Bielsko-Biała_Pride_frame_001.jpg")

loc_res = freak_matcher.localize_object(f0, f1, min_inliers=10)
print(f"Hamming Matches: {loc_res['num_matches']} | Verified Inliers: {loc_res['num_inliers']} | Object Detected: {loc_res['detected']}")

if loc_res['detected']:
    scene_annotated = f1.copy()
    pts = np.int32(loc_res['dst_corners']).reshape(-1, 1, 2)
    cv2.polylines(scene_annotated, [pts], True, (255, 128, 0), 4, cv2.LINE_AA)

    matches_img = cv2.drawMatches(
        f0, loc_res['kp_template'],
        scene_annotated, loc_res['kp_scene'],
        loc_res['good_matches'][:30], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    show_rgb(matches_img, title=f"FREAK Hamming Matching & Object Localization ({loc_res['num_inliers']} Inliers)")



## Step 5: Save FREAK Model


In [ ]:
freak_bovw.save_model("freak_bovw_model.pkl")
sz = Path("freak_bovw_model.pkl").stat().st_size / (1024 * 1024)
print(f"Saved FREAK Model: freak_bovw_model.pkl ({sz:.2f} MB)")

